# Guided Beta-Model Replication

This notebook uses the same guided JSON helper workflow as `qmm_guided_json_runner.ipynb`, but it is preconfigured to reproduce the beta-equilibrium comparison figure with:

1. van der Waals with `b_{pn} = b_n`
2. Carnahan-Starling with `b_{pn} = b_n`
3. TVM with `b_{pn} = b_n`
4. Carnahan-Starling with `b_{pn} \neq b_n`
5. TVM with `b_{pn} \neq b_n`

The notebook builds the configs with the guided helper, runs them if needed, loads the generated beta-equilibrium CSVs, and then makes the four-panel comparison plot.

In [ ]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

candidate_roots = []
seen = set()
for base in [Path.cwd(), *Path.cwd().parents]:
    for candidate in (base, base / 'QMM'):
        resolved = candidate.resolve()
        marker = str(resolved)
        if marker not in seen:
            seen.add(marker)
            candidate_roots.append(resolved)

ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'qmm').exists() and (candidate / 'examples').exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError('Could not find the QMM project root from this notebook.')

SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from qmm.constants import DEFAULT_PHYSICAL_CONSTANTS
from qmm.qmm_json_helper import (
    build_configs,
    choose_python,
    default_request,
    python_details,
    run_configs,
    save_configs,
    summarize_run,
)

PYTHON_BIN = choose_python(ROOT)
PYTHON_INFO = python_details(ROOT, PYTHON_BIN)
NOTEBOOK_OUTPUT_DIR = ROOT / 'results' / 'generated' / 'guided_beta_model_replication'
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'QMM root: {ROOT}')
print(f'Runner Python: {PYTHON_BIN}')
print(f'Runner version: {PYTHON_INFO["version"]}')
print(f'Matplotlib available for runner: {PYTHON_INFO["has_matplotlib"]}')
print(f'Notebook output directory: {NOTEBOOK_OUTPUT_DIR.relative_to(ROOT)}')

In [ ]:
BASE_PREFIX = 'guided_beta_model_replication'
FORCE_RERUN = False
RERUN_IF_SOURCE_NEWER = True
MUON_THRESHOLD = 1.0e-12
SOURCE_DEPENDENCIES = [
    ROOT / 'src' / 'qmm' / 'asymmetry.py',
    ROOT / 'src' / 'qmm' / 'quarkyonic.py',
    ROOT / 'src' / 'qmm' / 'sound_speed.py',
    ROOT / 'src' / 'qmm' / 'qmm_json_helper.py',
]

MODEL_REQUEST_SPECS = [
    {
        'run_name': f'{BASE_PREFIX}_vdw_equal_b',
        'output_subdir': f'{BASE_PREFIX}_vdw_equal_b',
        'mean_field': 'vdw',
        'excluded_volume': 'vdw',
        'model_name': 'vdw',
        'branch_mode': 'equal_b',
    },
    {
        'run_name': f'{BASE_PREFIX}_cs',
        'output_subdir': f'{BASE_PREFIX}_cs',
        'mean_field': 'vdw',
        'excluded_volume': 'cs',
        'model_name': 'cs',
        'branch_mode': 'both',
    },
    {
        'run_name': f'{BASE_PREFIX}_tvm',
        'output_subdir': f'{BASE_PREFIX}_tvm',
        'mean_field': 'vdw',
        'excluded_volume': 'tvm',
        'model_name': 'tvm',
        'branch_mode': 'both',
    },
]

def guided_request_from_spec(spec):
    request = default_request()
    request.update(
        {
            'run_name': spec['run_name'],
            'output_subdir': spec['output_subdir'],
            'mean_field': spec['mean_field'],
            'excluded_volume': spec['excluded_volume'],
            'model_name': spec['model_name'],
            'preset': 'asymmetric_beta',
            'parameter_mode': 'none',
            'parameter_value': None,
            'target_k0': None,
            'quick_mode': False,
            'momentum_mode': 'quarkyonic',
            'lambda_momentum_mev': 200.0,
            'branch_mode': spec['branch_mode'],
            'beta_equilibrium': True,
            'proton_fraction_values': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],
            'write_json': True,
            'write_csv': True,
            'write_plots': False,
            'plot_formats': [],
        }
    )
    return request

requests = [guided_request_from_spec(spec) for spec in MODEL_REQUEST_SPECS]
for request in requests:
    print(json.dumps(request, indent=2))
    print()

In [ ]:
configs = []
for request in requests:
    configs.extend(build_configs(ROOT, request, PYTHON_BIN))

for config in configs:
    config['output']['write_plots'] = False
    config['output']['plot_formats'] = []
    config['quarkyonic']['lambda_momentum_mev'] = 200.0
    config['quarkyonic']['n_min_ratio'] = 0.05
    config['quarkyonic']['n_max_ratio'] = 5.0
    config['quarkyonic']['n_points'] = 120
    config['quarkyonic']['fq_scan_points'] = 121
    config['quarkyonic']['shell_integral_points'] = 400
    config['quarkyonic']['quark_integral_points'] = 400
    config['quarkyonic']['smoothing_window'] = 9
    config['quarkyonic']['smoothing_degree'] = 3
    config['asymmetric']['beta_equilibrium'] = True

print(f'Generated {len(configs)} config(s):')
for config in configs:
    print(f"- {config['run_name']} | model={config['model']['name']} | branch={config['asymmetric']['branch_mode']}")

config_paths = save_configs(ROOT, configs)
for path in config_paths:
    print(f'Saved config: {path.relative_to(ROOT)}')

In [ ]:
def summary_path_for_config(config):
    return ROOT / config['output']['directory'] / f"{config['run_name']}_summary.json"

def summaries_stale(paths):
    if not RERUN_IF_SOURCE_NEWER:
        return False
    existing = [path for path in paths if path.exists()]
    sources = [path for path in SOURCE_DEPENDENCIES if path.exists()]
    if not existing or not sources:
        return False
    newest_source = max(path.stat().st_mtime for path in sources)
    oldest_summary = min(path.stat().st_mtime for path in existing)
    return newest_source > oldest_summary

summary_paths = [summary_path_for_config(config) for config in configs]
need_run = FORCE_RERUN or not all(path.exists() for path in summary_paths) or summaries_stale(summary_paths)

if need_run:
    run_results = run_configs(ROOT, config_paths, PYTHON_BIN)
else:
    run_results = []
    for path in summary_paths:
        run_results.append(
            {
                'command': None,
                'stdout': '',
                'stderr': '',
                'summary': json.loads(path.read_text()),
                'summary_path': path,
            }
        )

for result in run_results:
    print('Command used:' if result['command'] else 'Loaded existing summary:')
    print(result['command'] or result['summary_path'].relative_to(ROOT))
    print()
    print(summarize_run(result['summary']))
    print()

In [ ]:
def resolve_output_path(path_text):
    path = Path(path_text)
    return path if path.is_absolute() else ROOT / path


def load_beta_curve(path):
    with path.open('r', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    curve = {}
    for key in ['n_b', 'n_over_n0', 'y', 'quark_fraction', 'energy_density', 'pressure', 'vs2', 'mu_e', 'eps_mu']:
        curve[key] = np.array([
            float(row[key]) if row[key] not in {'', 'nan', 'None'} else np.nan
            for row in rows
        ], dtype=float)
    order = np.argsort(curve['n_over_n0'])
    for key in curve:
        curve[key] = curve[key][order]
    return curve


def branch_key(summary):
    fit = summary['asymmetric_fit']
    return fit['model'], fit['branch_mode']


curve_results = {}
fit_rows = []
for result in run_results:
    summary = result['summary']
    key = branch_key(summary)
    beta_csv = resolve_output_path(summary['outputs']['asymmetric_beta_csv'])
    curve_results[key] = {
        'summary': summary,
        'curve': load_beta_curve(beta_csv),
        'beta_csv': beta_csv,
    }
    fit_rows.append(
        {
            'model': summary['asymmetric_fit']['model'],
            'branch_mode': summary['asymmetric_fit']['branch_mode'],
            'a_pn_over_a_n': round(float(summary['asymmetric_fit']['ratio_a_pn_over_a_n']), 6),
            'b_pn_over_b_n': round(float(summary['asymmetric_fit']['ratio_b_pn_over_b_n']), 6),
            'J': round(float(summary['asymmetric_fit']['J']), 6),
            'L': round(float(summary['asymmetric_fit']['L']), 6),
            'K0': round(float(summary['asymmetric_fit']['K0']), 6),
        }
    )

for key, item in curve_results.items():
    print(f'{key}: {item["beta_csv"].relative_to(ROOT)}')

try:
    import pandas as pd
    display(pd.DataFrame(fit_rows).sort_values(['branch_mode', 'model']).reset_index(drop=True))
except Exception:
    from pprint import pprint
    pprint(fit_rows)

In [ ]:
plt.rcParams.update(
    {
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 12,
        'legend.fontsize': 9,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'axes.spines.top': True,
        'axes.spines.right': True,
    }
)

PLOT_ORDER = [
    ('vdw', 'equal_b'),
    ('cs', 'equal_b'),
    ('tvm', 'equal_b'),
    ('cs', 'target_l'),
    ('tvm', 'target_l'),
]

CURVE_STYLES = {
    ('vdw', 'equal_b'): {
        'label': 'van der Waals',
        'color': 'black',
        'linestyle': '-',
        'linewidth': 1.8,
    },
    ('cs', 'equal_b'): {
        'label': 'Carnahan-Starling',
        'color': '#3B61FF',
        'linestyle': '--',
        'linewidth': 1.8,
    },
    ('tvm', 'equal_b'): {
        'label': 'trivial-model',
        'color': '#FF6666',
        'linestyle': '-.',
        'linewidth': 1.8,
    },
    ('cs', 'target_l'): {
        'label': 'Carnahan-Starling',
        'color': '#4DAA57',
        'linestyle': ':',
        'linewidth': 1.8,
    },
    ('tvm', 'target_l'): {
        'label': 'trivial-model',
        'color': '#FF9C5A',
        'linestyle': (0, (5, 1.4, 1.2, 1.4)),
        'linewidth': 1.8,
    },
}


def energy_per_baryon_gev(curve):
    return curve['energy_density'] / curve['n_b'] / 1000.0 - DEFAULT_PHYSICAL_CONSTANTS.m_nucleon / 1000.0


def muon_intervals(curve):
    mask = np.isfinite(curve['eps_mu']) & (curve['eps_mu'] > MUON_THRESHOLD)
    if not np.any(mask):
        return []
    indices = np.where(mask)[0]
    breaks = np.where(np.diff(indices) > 1)[0]
    starts = np.concatenate(([indices[0]], indices[breaks + 1]))
    ends = np.concatenate((indices[breaks], [indices[-1]]))
    return [(int(start), int(end)) for start, end in zip(starts, ends)]


fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.2), sharex=True)
fig.subplots_adjust(left=0.10, right=0.98, bottom=0.11, top=0.97, wspace=0.18, hspace=0.14)

ax_y = axes[0, 0]
ax_e = axes[0, 1]
ax_fq = axes[1, 0]
ax_vs = axes[1, 1]

for key in PLOT_ORDER:
    curve = curve_results[key]['curve']
    style = CURVE_STYLES[key]
    x_values = curve['n_over_n0']
    y_values = curve['y']
    fq_values = curve['quark_fraction']
    energy_values = energy_per_baryon_gev(curve)
    vs2_values = curve['vs2']

    ax_y.plot(x_values, y_values, color=style['color'], ls=style['linestyle'], lw=style['linewidth'])
    ax_e.plot(x_values, energy_values, color=style['color'], ls=style['linestyle'], lw=style['linewidth'])
    ax_fq.plot(x_values, fq_values, color=style['color'], ls=style['linestyle'], lw=style['linewidth'])
    ax_vs.plot(x_values, vs2_values, color=style['color'], ls=style['linestyle'], lw=style['linewidth'])

    for start, end in muon_intervals(curve):
        if start > 0:
            ax_y.plot(
                x_values[start],
                y_values[start],
                marker='^',
                markersize=5.5,
                markerfacecolor='white',
                markeredgecolor=style['color'],
                markeredgewidth=0.9,
                linestyle='None',
                zorder=5,
            )
        if end < len(x_values) - 1:
            ax_y.plot(
                x_values[end],
                y_values[end],
                marker='^',
                markersize=5.5,
                markerfacecolor=style['color'],
                markeredgecolor=style['color'],
                markeredgewidth=0.9,
                linestyle='None',
                zorder=5,
            )

ax_y.set_xlim(0.0, 5.0)
ax_y.set_ylim(0.0, 0.20)
ax_y.set_ylabel(r'$y$')
ax_y.set_xlabel(r'$\rho_B\ [\rho_0]$')
ax_y.text(0.02, 0.94, '(a)', transform=ax_y.transAxes, fontsize=10)

ax_e.set_xlim(0.0, 5.0)
ax_e.set_ylim(0.0, 0.50)
ax_e.set_ylabel(r'$\epsilon / \rho_B - m_N$ [GeV]')
ax_e.set_xlabel(r'$\rho_B\ [\rho_0]$')
ax_e.text(0.96, 0.94, '(b)', transform=ax_e.transAxes, ha='right', fontsize=10)

equal_handles = [
    Line2D([0], [0], color=CURVE_STYLES[key]['color'], ls=CURVE_STYLES[key]['linestyle'], lw=CURVE_STYLES[key]['linewidth'])
    for key in [('vdw', 'equal_b'), ('cs', 'equal_b'), ('tvm', 'equal_b')]
]
equal_labels = [CURVE_STYLES[key]['label'] for key in [('vdw', 'equal_b'), ('cs', 'equal_b'), ('tvm', 'equal_b')]]
unequal_handles = [
    Line2D([0], [0], color=CURVE_STYLES[key]['color'], ls=CURVE_STYLES[key]['linestyle'], lw=CURVE_STYLES[key]['linewidth'])
    for key in [('cs', 'target_l'), ('tvm', 'target_l')]
]
unequal_labels = [CURVE_STYLES[key]['label'] for key in [('cs', 'target_l'), ('tvm', 'target_l')]]
legend_equal = ax_e.legend(equal_handles, equal_labels, title=r'$b_{pn}=b_n$:', loc='upper left', frameon=False, bbox_to_anchor=(0.02, 1.00), title_fontsize=10)
legend_unequal = ax_e.legend(unequal_handles, unequal_labels, title=r'$b_{pn}\neq b_n$:', loc='upper left', frameon=False, bbox_to_anchor=(0.02, 0.54), title_fontsize=10)
ax_e.add_artist(legend_equal)

ax_fq.set_xlim(0.0, 5.0)
ax_fq.set_ylim(0.0, 0.70)
ax_fq.set_ylabel(r'$f_q$')
ax_fq.set_xlabel(r'$\rho_B\ [\rho_0]$')
ax_fq.text(0.08, 0.12, r'$\Lambda = 0.2$ GeV', transform=ax_fq.transAxes, fontsize=10)
ax_fq.text(0.96, 0.06, '(c)', transform=ax_fq.transAxes, ha='right', fontsize=10)

ax_vs.set_xlim(0.0, 5.0)
ax_vs.set_ylim(0.0, 1.00)
ax_vs.set_ylabel(r'$v_s^2$')
ax_vs.set_xlabel(r'$\rho_B\ [\rho_0]$')
ax_vs.axhline(1.0 / 3.0, color='gray', ls=(0, (4, 4)), lw=0.9)
ax_vs.text(0.96, 0.94, '(d)', transform=ax_vs.transAxes, ha='right', fontsize=10)

for axis in axes.flat:
    axis.grid(alpha=0.18, lw=0.6)
    axis.tick_params(axis='both', labelsize=10)

figure_png = NOTEBOOK_OUTPUT_DIR / 'guided_beta_model_replication.png'
figure_pdf = NOTEBOOK_OUTPUT_DIR / 'guided_beta_model_replication.pdf'
fig.savefig(figure_png, dpi=240)
fig.savefig(figure_pdf)
plt.show()

print(f'Saved: {figure_png.relative_to(ROOT)}')
print(f'Saved: {figure_pdf.relative_to(ROOT)}')

In [ ]:
summary_rows = []
for key in PLOT_ORDER:
    curve = curve_results[key]['curve']
    fit = curve_results[key]['summary']['asymmetric_fit']
    intervals = muon_intervals(curve)
    appearance = float(curve['n_over_n0'][intervals[0][0]]) if intervals and intervals[0][0] > 0 else np.nan
    disappearance = float(curve['n_over_n0'][intervals[-1][1]]) if intervals and intervals[-1][1] < len(curve['n_over_n0']) - 1 else np.nan
    summary_rows.append(
        {
            'model': key[0],
            'branch_mode': key[1],
            'J': round(float(fit['J']), 6),
            'L': round(float(fit['L']), 6),
            'K0': round(float(fit['K0']), 6),
            'muon_appearance_n_over_n0': round(appearance, 6) if np.isfinite(appearance) else np.nan,
            'muon_disappearance_n_over_n0': round(disappearance, 6) if np.isfinite(disappearance) else np.nan,
            'y_max': round(float(np.nanmax(curve['y'])), 6),
            'fq_max': round(float(np.nanmax(curve['quark_fraction'])), 6),
            'vs2_max': round(float(np.nanmax(curve['vs2'])), 6),
        }
    )

summary_csv = NOTEBOOK_OUTPUT_DIR / 'guided_beta_model_replication_summary.csv'
with summary_csv.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except Exception:
    from pprint import pprint
    pprint(summary_rows)

print(f'Saved: {summary_csv.relative_to(ROOT)}')